# Checkpoint 38: Temporal Dataset Validation

This notebook reviews the three historical backtesting snapshots and the separate current active-workforce scoring population.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VALIDATION_DIR = PROCESSED_DIR / "temporal_dataset_validation"

## Load the generated datasets

In [ ]:
historical = pd.read_csv(
    PROCESSED_DIR / "retention_multi_snapshot.csv",
    parse_dates=["snapshot_date", "prediction_end_date"],
)
current = pd.read_csv(
    PROCESSED_DIR / "current_active_scoring_population.csv",
    parse_dates=["snapshot_date", "prediction_end_date"],
)
checks = pd.read_csv(VALIDATION_DIR / "validation_checks.csv")
summary = pd.read_csv(
    VALIDATION_DIR / "snapshot_summary.csv",
    parse_dates=["snapshot_date", "prediction_end_date"],
)
panel = pd.read_csv(VALIDATION_DIR / "panel_summary.csv")

print("Historical shape:", historical.shape)
print("Current shape:", current.shape)

## Validation checks

Every row below should show `PASS`.

In [ ]:
checks

## Snapshot populations and outcomes

In [ ]:
summary

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    summary["snapshot_date"].dt.strftime("%Y-%m-%d"),
    summary["eligible_employees"],
    color="#4C78A8",
)
axes[0].set_title("Eligible employees by snapshot")
axes[0].set_ylabel("Employee snapshots")
axes[0].tick_params(axis="x", rotation=25)

axes[1].plot(
    summary["snapshot_date"],
    summary["positive_rate"] * 100,
    marker="o",
    color="#E45756",
)
axes[1].set_title("Twelve-month attrition rate")
axes[1].set_ylabel("Positive rate (%)")
axes[1].set_xlabel("Snapshot date")

figure.tight_layout()
plt.show()

## Repeated employees

The same active employee may appear at several dates. This is expected in a temporal panel, but later grouped validation must prevent employee overlap when an employee-disjoint test is required.

In [ ]:
employee_snapshot_counts = historical.groupby("employee_id")["snapshot_date"].nunique()
employee_snapshot_counts.value_counts().sort_index().rename_axis(
    "number_of_snapshots"
).to_frame("employees")

## Missing-value review

Missing performance values usually mean that no review existed by that historical snapshot. Missing training scores usually mean that no completed scored program existed during the trailing year.

In [ ]:
(
    historical.isna().mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .head(15)
    .to_frame()
)

## Current scoring separation

Current employees have features but no known future attrition target.

In [ ]:
pd.DataFrame(
    {
        "population": ["Historical", "Current scoring"],
        "rows": [len(historical), len(current)],
        "known_targets": [
            historical["attrition_next_12m"].notna().sum(),
            current["attrition_next_12m"].notna().sum(),
        ],
    }
)

## Conclusion

The temporal panel is ready for feature redundancy analysis in Checkpoint 39 and train/validation/test design in Checkpoint 40.